# 01 - Build dataset (sezon 2024)

Lecimy pętlą po wszystkich wyścigach sezonu 2024 i budujemy **dwa** datasety:

- `laps_clean.parquet` - jeden wiersz = jedno czyste okrazenie (pod **regresje** czasu)
- `driver_race.parquet` - jeden wiersz = jeden kierowca w jednym wyscigu (pod **klasyfikacje** podium)

Cache i pobieranie obsluguje FastF1 (jak w `00-data-load`). Pierwsze odpalenie sciaga
sporo danych - moze potrwac kilka/kilkanascie minut.

In [1]:
import fastf1
import pandas as pd
import numpy as np
import os

os.makedirs('../cache', exist_ok=True)
os.makedirs('../data', exist_ok=True)
fastf1.Cache.enable_cache('../cache')

SEASON = 2024

## Lista rund (tylko wyscigi)

In [2]:
schedule = fastf1.get_event_schedule(SEASON, include_testing=False)
rounds = schedule[['RoundNumber', 'EventName', 'EventFormat']].copy()
rounds = rounds[rounds['RoundNumber'] > 0].reset_index(drop=True)
print(f'{len(rounds)} rund w sezonie {SEASON}')
rounds

24 rund w sezonie 2024


,RoundNumber,EventName,EventFormat
0,1,Bahrain Grand Prix,conventional
1,2,Saudi Arabian Grand Prix,conventional
2,3,Australian Grand Prix,conventional
3,4,Japanese Grand Prix,conventional
4,5,Chinese Grand Prix,sprint_qualifying
5,6,Miami Grand Prix,sprint_qualifying
6,7,Emilia Romagna Grand Prix,conventional
7,8,Monaco Grand Prix,conventional
8,9,Canadian Grand Prix,conventional
9,10,Spanish Grand Prix,conventional


## Petla po wyscigach

Dla kazdego GP zbieramy:
- **laps** (+ pogoda zlaczona po czasie, + nr rundy / nazwa)
- **results** (pozycja koncowa, pozycja startowa, zespol, czasy kwalifikacji)

Pogode i `pick_quicklaps` trzeba robic na obiekcie `Laps` *przed* sklejeniem,
bo po `pd.concat` zostaje zwykly DataFrame bez tych metod. Tu robimy tylko
zlaczenie pogody w petli; wlasciwe czyszczenie - nizej, na calosci.

In [3]:
all_laps = []
all_results = []

for _, row in rounds.iterrows():
    rnd = int(row['RoundNumber'])
    name = row['EventName']
    try:
        s = fastf1.get_session(SEASON, rnd, 'R')
        s.load()  # timing + laps + weather + results
    except Exception as e:
        print(f'  [skip] R{rnd} {name}: {e}')
        continue

    # --- laps + weather ---
    laps = s.laps
    if laps is None or len(laps) == 0:
        print(f'  [skip] R{rnd} {name}: brak laps')
        continue
    weather = laps.get_weather_data()
    laps = laps.reset_index(drop=True)
    weather = weather.reset_index(drop=True)
    laps = pd.concat([laps, weather.drop(columns=['Time'])], axis=1)
    laps['Round'] = rnd
    laps['EventName'] = name
    all_laps.append(laps)

    # --- results ---
    res = s.results.copy()
    res['Round'] = rnd
    res['EventName'] = name
    all_results.append(res)

    print(f'  [ok]   R{rnd:>2} {name:<28} laps={len(laps):>4}')

laps_raw = pd.concat(all_laps, ignore_index=True)
results_raw = pd.concat(all_results, ignore_index=True)
print('\nlaps_raw:', laps_raw.shape, '| results_raw:', results_raw.shape)

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

  [ok]   R 1 Bahrain Grand Prix           laps=1129


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
_api        WARNING 	Failed to align laps for driver

  [ok]   R 2 Saudi Arabian Grand Prix     laps= 901


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

...

  [ok]   R22 Las Vegas Grand Prix         laps= 938


req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing

  [ok]   R23 Qatar Grand Prix             laps= 943


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

  [ok]   R24 Abu Dhabi Grand Prix         laps=1035

laps_raw: (26604, 40) | results_raw: (479, 24)


## Dataset 1: `laps_clean` (pod regresje)

Czyszczenie:
1. usun okrazenia bez `LapTime`
2. usun in/out-lapy (niepuste `PitInTime`/`PitOutTime`)
3. zostaw tylko zielona flage `TrackStatus == '1'` (bez SC/VSC/zoltych)
4. zostaw `IsAccurate == True`
5. odrzuc odstajace: w obrebie wyscigu zostaw okrazenia <= 107% najszybszego

Wyrzucamy tez kolumny, ktore bylyby **leakage** (sektory i predkosci policzone
z tego samego okrazenia - model by przez nie 'sciagal' odpowiedz).

In [4]:
clean = laps_raw.copy()
clean = clean[clean['LapTime'].notna()]
clean = clean[clean['PitInTime'].isna() & clean['PitOutTime'].isna()]
clean = clean[clean['TrackStatus'].astype(str) == '1']
clean = clean[clean['IsAccurate'] == True]
clean['LapTime_s'] = clean['LapTime'].dt.total_seconds()

# regula 107% per wyscig (odciecie outlierow: traffic, lift&coast itp.)
fastest = clean.groupby('Round')['LapTime_s'].transform('min')
clean = clean[clean['LapTime_s'] <= 1.07 * fastest]

# kolumny do modelowania (reszta - w tym leakage - leci)
keep = ['Round', 'EventName', 'Driver', 'Team', 'LapNumber', 'Stint',
        'Compound', 'TyreLife', 'FreshTyre',
        'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall',
        'LapTime_s']
laps_clean = clean[keep].copy()
laps_clean['FreshTyre'] = laps_clean['FreshTyre'].astype(bool)

print('laps_clean:', laps_clean.shape)
print('okrazen na wyscig (mediana):', int(laps_clean.groupby('Round').size().median()))
laps_clean.head()

laps_clean: (20603, 16)
okrazen na wyscig (mediana): 861


C:\Users\jaku2\AppData\Local\Temp\ipykernel_42176\662964356.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean['LapTime_s'] = clean['LapTime'].dt.total_seconds()


,Round,EventName,Driver,Team,LapNumber,Stint,Compound,TyreLife,FreshTyre,AirTemp,TrackTemp,Humidity,Pressure,WindSpeed,Rainfall,LapTime_s
1,1,Bahrain Grand Prix,VER,Red Bull Racing,2.0,1.0,SOFT,5.0,False,18.3,23.8,49.0,1017.0,1.1,False,96.296
2,1,Bahrain Grand Prix,VER,Red Bull Racing,3.0,1.0,SOFT,6.0,False,18.3,23.8,49.0,1017.1,1.5,False,96.753
3,1,Bahrain Grand Prix,VER,Red Bull Racing,4.0,1.0,SOFT,7.0,False,18.3,23.7,49.0,1017.0,0.9,False,96.647
4,1,Bahrain Grand Prix,VER,Red Bull Racing,5.0,1.0,SOFT,8.0,False,18.3,23.5,50.0,1017.1,1.5,False,97.173
5,1,Bahrain Grand Prix,VER,Red Bull Racing,6.0,1.0,SOFT,9.0,False,18.3,23.5,50.0,1017.1,1.1,False,97.092


In [5]:
laps_clean.to_parquet('../data/laps_clean.parquet', index=False)
print('zapisano ../data/laps_clean.parquet')

zapisano ../data/laps_clean.parquet


## Dataset 2: `driver_race` (pod klasyfikacje podium)

Jeden wiersz = jeden kierowca w jednym wyscigu. Target: **podium** = pozycja koncowa <= 3.

Cechy trzymamy jako **przedwyscigowe** (znane przed startem) - zeby to bylo uczciwe
*przewidywanie*, a nie opis po fakcie:
- `GridPosition` - pozycja startowa (laczy wynik kwalifikacji + kary)
- `Team` - proxy sily konstruktora
- `quali_gap_s` - strata do pole position w kwalifikacjach (sekundy)

(Cechy z tempa wyscigowego celowo pomijam - znamy je dopiero po wyscigu. Jak chcesz
wariant 'opisowy', mozna domieszac mediane tempa z `laps_clean`.)

In [6]:
res = results_raw.copy()

# najlepszy czas kwalifikacji = min(Q1,Q2,Q3) w sekundach
for q in ['Q1', 'Q2', 'Q3']:
    res[q + '_s'] = pd.to_timedelta(res[q]).dt.total_seconds()
res['quali_best_s'] = res[['Q1_s', 'Q2_s', 'Q3_s']].min(axis=1)

# strata do pole w obrebie wyscigu
pole = res.groupby('Round')['quali_best_s'].transform('min')
res['quali_gap_s'] = res['quali_best_s'] - pole

dr = res[['Round', 'EventName', 'Abbreviation', 'TeamName',
          'GridPosition', 'Position', 'Points', 'Status', 'quali_gap_s']].copy()
dr = dr.rename(columns={'Abbreviation': 'Driver', 'TeamName': 'Team'})

# start z alei serwisowej bywa kodowany jako 0 -> traktuj jak ostatnie pole
dr['GridPosition'] = dr['GridPosition'].replace(0, np.nan)
dr['GridPosition'] = dr['GridPosition'].fillna(dr.groupby('Round')['GridPosition'].transform('max') + 1)

# target
dr = dr[dr['Position'].notna()].copy()
dr['podium'] = (dr['Position'] <= 3).astype(int)

print('driver_race:', dr.shape)
print('podium rate:', round(dr['podium'].mean(), 3))
dr.head()

driver_race: (479, 10)
podium rate: 0.15


,Round,EventName,Driver,Team,GridPosition,Position,Points,Status,quali_gap_s,podium
0,1,Bahrain Grand Prix,VER,Red Bull Racing,1.0,1.0,26.0,Finished,NaN,1
1,1,Bahrain Grand Prix,PER,Red Bull Racing,5.0,2.0,18.0,Finished,NaN,1
2,1,Bahrain Grand Prix,SAI,Ferrari,4.0,3.0,15.0,Finished,NaN,1
3,1,Bahrain Grand Prix,LEC,Ferrari,2.0,4.0,12.0,Finished,NaN,0
4,1,Bahrain Grand Prix,RUS,Mercedes,3.0,5.0,10.0,Finished,NaN,0


In [7]:
dr.to_parquet('../data/driver_race.parquet', index=False)
print('zapisano ../data/driver_race.parquet')

zapisano ../data/driver_race.parquet
